In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # Balanced MRI数据分析与处理
# 
# 分析balanced_output文件夹中的平衡数据集，包括：
# - balanced_data_4d10000.nii.gz (4D影像数据)
# - balanced_labels_3d10000.nii.gz (3D标签数据)

# ## 1. 导入必要的库

import nibabel as nib
import numpy as np
import pandas as pd
from pathlib import Path
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time
from datetime import datetime
from tqdm import tqdm

# 设置显示选项
np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 8)

# ## 2. 设置数据路径和扫描balanced数据

# 根目录路径
ROOT_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS")

def scan_balanced_dataset(root_dir):
    """
    扫描所有balanced_output文件夹，检查数据完整性
    
    Parameters:
    -----------
    root_dir : Path
        数据集根目录路径
    
    Returns:
    --------
    dict : 扫描结果
    """
    root_dir = Path(root_dir)
    
    print(f"📁 扫描目录: {root_dir}")
    print("=" * 80)
    
    # 定义balanced数据的文件名
    balanced_files = {
        "4D_balanced": "balanced_data_4d10000.nii.gz",
        "3D_balanced_labels": "balanced_labels_3d10000.nii.gz"
    }
    
    results = {
        "scan_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "root_directory": str(root_dir),
        "total_subjects": 0,
        "valid_subjects": 0,
        "incomplete_subjects": 0,
        "subjects": []
    }
    
    # 查找所有FOR_开头的文件夹
    subject_folders = sorted([f for f in root_dir.iterdir() 
                            if f.is_dir() and f.name.startswith("FOR_")])
    
    print(f"🔍 找到 {len(subject_folders)} 个受试者文件夹\n")
    
    # 检查每个受试者的balanced_output文件夹
    for idx, subject_folder in enumerate(subject_folders, 1):
        balanced_dir = subject_folder / "balanced_output"
        
        subject_info = {
            "subject_id": subject_folder.name,
            "path": str(subject_folder),
            "balanced_dir": str(balanced_dir),
            "status": "完整",
            "missing_files": [],
            "existing_files": {},
            "all_files_in_balanced": []
        }
        
        print(f"[{idx}/{len(subject_folders)}] 检查受试者: {subject_folder.name}")
        
        # 检查balanced_output文件夹是否存在
        if not balanced_dir.exists():
            print(f"  ✗ balanced_output文件夹不存在")
            subject_info["status"] = "缺失文件夹"
            results["incomplete_subjects"] += 1
        else:
            # 列出balanced_output中的所有文件
            all_files = list(balanced_dir.glob("*"))
            subject_info["all_files_in_balanced"] = [f.name for f in all_files]
            print(f"  📂 找到balanced_output文件夹，包含 {len(all_files)} 个文件")
            
            # 显示所有文件
            for f in all_files:
                file_size_mb = f.stat().st_size / (1024 * 1024)
                print(f"     - {f.name} ({file_size_mb:.2f} MB)")
            
            # 检查必需的balanced文件
            all_files_exist = True
            for file_type, file_name in balanced_files.items():
                file_path = balanced_dir / file_name
                
                if file_path.exists():
                    subject_info["existing_files"][file_type] = str(file_path)
                    file_size_mb = file_path.stat().st_size / (1024 * 1024)
                    subject_info["existing_files"][f"{file_type}_size_mb"] = round(file_size_mb, 2)
                    print(f"  ✓ {file_type}: 找到 ({file_size_mb:.2f} MB)")
                else:
                    all_files_exist = False
                    subject_info["missing_files"].append(file_type)
                    subject_info["status"] = "不完整"
                    print(f"  ✗ {file_type}: 缺失")
            
            if all_files_exist:
                results["valid_subjects"] += 1
                print(f"  状态: ✅ 完整\n")
            else:
                results["incomplete_subjects"] += 1
                print(f"  状态: ⚠️ 不完整\n")
        
        results["subjects"].append(subject_info)
    
    results["total_subjects"] = len(subject_folders)
    
    return results

# 执行扫描
print("🚀 开始扫描Balanced数据集...\n")
scan_results = scan_balanced_dataset(ROOT_DIR)

# ## 3. 显示扫描结果汇总

print("\n" + "=" * 80)
print("📊 Balanced数据集汇总")
print("=" * 80)
print(f"扫描时间: {scan_results['scan_time']}")
print(f"根目录: {scan_results['root_directory']}")
print(f"\n📈 统计信息:")
print(f"  • 总受试者数: {scan_results['total_subjects']}")
print(f"  • 完整数据受试者数: {scan_results['valid_subjects']} ({scan_results['valid_subjects']/max(scan_results['total_subjects'], 1)*100:.1f}%)")
print(f"  • 不完整数据受试者数: {scan_results['incomplete_subjects']} ({scan_results['incomplete_subjects']/max(scan_results['total_subjects'], 1)*100:.1f}%)")

# 保存扫描结果
output_dir = Path("./balanced_analysis_results")
output_dir.mkdir(exist_ok=True)

scan_results_file = output_dir / f"balanced_scan_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(scan_results_file, 'w', encoding='utf-8') as f:
    json.dump(scan_results, f, ensure_ascii=False, indent=2)

print(f"\n💾 扫描结果已保存到: {scan_results_file}")

# ## 4. 获取有效的balanced数据列表

valid_balanced_subjects = []
for subject in scan_results["subjects"]:
    if subject["status"] == "完整":
        valid_balanced_subjects.append({
            "subject_id": subject["subject_id"],
            "path": subject["path"],
            "balanced_4d_path": subject["existing_files"]["4D_balanced"],
            "balanced_3d_labels_path": subject["existing_files"]["3D_balanced_labels"],
            "balanced_dir": subject["balanced_dir"]
        })

print(f"\n✅ 找到 {len(valid_balanced_subjects)} 个数据完整的受试者")

if len(valid_balanced_subjects) > 0:
    print("\n完整数据的受试者列表:")
    for i, subject in enumerate(valid_balanced_subjects[:5], 1):
        print(f"{i}. {subject['subject_id']}")
    if len(valid_balanced_subjects) > 5:
        print(f"... 及其他 {len(valid_balanced_subjects)-5} 个受试者")

# ## 5. 深入分析第一个受试者的balanced数据

if valid_balanced_subjects:
    first_subject = valid_balanced_subjects[0]
    print(f"\n📊 深入分析受试者: {first_subject['subject_id']}")
    print("=" * 80)
    
    # 加载balanced数据
    print("\n🔄 加载Balanced NIfTI文件...")
    
    # 4D balanced数据
    balanced_4d_path = Path(first_subject['balanced_4d_path'])
    balanced_img_4d = nib.load(balanced_4d_path)
    balanced_data_4d = balanced_img_4d.get_fdata()
    
    print(f"✓ 4D balanced数据加载完成: {balanced_4d_path.name}")
    
    # 3D balanced标签
    balanced_label_3d_path = Path(first_subject['balanced_3d_labels_path'])
    balanced_label_3d = nib.load(balanced_label_3d_path)
    balanced_data_label = balanced_label_3d.get_fdata()
    
    print(f"✓ 3D balanced标签加载完成: {balanced_label_3d_path.name}")

# ## 6. 检查balanced数据的形状和特征

print("\n📐 Balanced数据形状检查:")
print("=" * 80)

# 4D数据形状
print(f"\n4D Balanced数据:")
print(f"  • 实际形状: {balanced_data_4d.shape}")
print(f"  • 维度说明: (X={balanced_data_4d.shape[0]}, Y={balanced_data_4d.shape[1]}, Z={balanced_data_4d.shape[2]}, Modalities={balanced_data_4d.shape[3]})")
print(f"  • 数据类型: {balanced_data_4d.dtype}")
print(f"  • 内存占用: {balanced_data_4d.nbytes / (1024**3):.2f} GB")

# 3D标签形状
print(f"\n3D Balanced标签:")
print(f"  • 实际形状: {balanced_data_label.shape}")
print(f"  • 数据类型: {balanced_data_label.dtype}")
print(f"  • 内存占用: {balanced_data_label.nbytes / (1024**2):.2f} MB")

# 检查形状匹配
spatial_match = (balanced_data_4d.shape[:3] == balanced_data_label.shape)
print(f"\n⚡ 空间维度匹配: {'✅ 是' if spatial_match else '❌ 否'}")

# 检查数据命名中的10000是什么含义
print("\n📝 文件名分析:")
print(f"  • 文件名包含 '10000' - 可能表示采样数量或其他参数")
total_voxels = np.prod(balanced_data_label.shape)
print(f"  • 总体素数: {total_voxels:,}")
print(f"  • 10000占总体素的比例: {10000/total_voxels*100:.4f}%")

# ## 7. 分析balanced数据的统计特征

print("\n📊 Balanced数据统计分析:")
print("=" * 80)

# 计算每个模态的统计信息
modality_stats = []
for m in range(balanced_data_4d.shape[3]):
    volume = balanced_data_4d[:, :, :, m]
    stats = {
        '模态': m,
        '最小值': np.min(volume),
        '最大值': np.max(volume),
        '均值': np.mean(volume),
        '标准差': np.std(volume),
        '中位数': np.median(volume),
        '非零比例': (np.count_nonzero(volume) / volume.size) * 100
    }
    modality_stats.append(stats)

df_modality_stats = pd.DataFrame(modality_stats)

print("\n各模态统计信息（前10个）:")
print(df_modality_stats.head(10).to_string(index=False))

# 检查数据标准化状态
print("\n🔍 检查数据标准化状态:")
for m in range(min(5, balanced_data_4d.shape[3])):
    volume = balanced_data_4d[:, :, :, m]
    mean_val = np.mean(volume)
    std_val = np.std(volume)
    print(f"  模态{m}: 均值={mean_val:.4f}, 标准差={std_val:.4f}")

# ## 8. 分析balanced标签分布

print("\n🏷️ Balanced标签分析:")
print("=" * 80)

# 获取唯一标签值
unique_labels = np.unique(balanced_data_label)
print(f"\n发现 {len(unique_labels)} 个唯一标签值")

# 统计每个标签的体素数量
label_counts = []
for label in unique_labels[:20]:  # 只显示前20个
    count = np.sum(balanced_data_label == label)
    percentage = (count / balanced_data_label.size) * 100
    label_counts.append({
        '标签值': int(label),
        '体素数量': count,
        '占比(%)': percentage
    })

df_balanced_labels = pd.DataFrame(label_counts)
df_balanced_labels = df_balanced_labels.sort_values('体素数量', ascending=False)

print("\n标签分布统计（按数量排序，前20个）:")
print(df_balanced_labels.to_string(index=False))

# 检查标签值范围
print(f"\n标签值范围: {int(unique_labels.min())} - {int(unique_labels.max())}")
print(f"标签值是否连续: {'是' if len(unique_labels) == (unique_labels.max() - unique_labels.min() + 1) else '否'}")

# 检查背景标签
background_count = np.sum(balanced_data_label == 0)
background_percentage = (background_count / balanced_data_label.size) * 100
print(f"\n背景体素（标签=0）:")
print(f"  • 数量: {background_count:,}")
print(f"  • 占比: {background_percentage:.2f}%")

# ## 9. 对比原始数据和balanced数据（如果原始数据存在）

print("\n🔄 对比原始数据和Balanced数据:")
print("=" * 80)

try:
    # 尝试加载原始数据进行对比
    original_4d_path = Path(first_subject['path']) / "evaluated/realigned_coregistered/nibabel_stacked_normalized_skull_stripped.nii.gz"
    original_3d_path = Path(first_subject['path']) / "seg/converted_alex_labels/resampled_synthseg_t1_mp2rage_alex_labels.nii.gz"
    
    if original_4d_path.exists() and original_3d_path.exists():
        print("✓ 找到原始数据，进行对比分析...")
        
        original_img = nib.load(original_4d_path)
        original_label = nib.load(original_3d_path)
        
        print(f"\n形状对比:")
        print(f"  原始4D数据: {original_img.shape}")
        print(f"  Balanced 4D数据: {balanced_data_4d.shape}")
        print(f"  原始3D标签: {original_label.shape}")
        print(f"  Balanced 3D标签: {balanced_data_label.shape}")
        
        if original_img.shape == balanced_img_4d.shape:
            print("\n  ⚠️ 形状相同 - balanced数据可能是重采样而非降采样")
        else:
            print("\n  ✅ 形状不同 - balanced数据已经过处理")
            
            # 计算降采样比例
            original_voxels = np.prod(original_img.shape[:3])
            balanced_voxels = np.prod(balanced_data_4d.shape[:3])
            reduction_ratio = balanced_voxels / original_voxels * 100
            print(f"  体素数量减少: {reduction_ratio:.2f}%")
    else:
        print("✗ 未找到原始数据进行对比")
        
except Exception as e:
    print(f"✗ 对比分析失败: {str(e)}")

# ## 10. 可视化balanced数据的切片

def plot_balanced_slices(data_4d, label_3d, subject_id, slice_idx=None):
    """
    可视化balanced数据的切片
    """
    if slice_idx is None:
        slice_idx = data_4d.shape[2] // 2
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # 显示前4个模态
    for i in range(4):
        if i < data_4d.shape[3]:
            im = axes[0, i].imshow(data_4d[:, :, slice_idx, i].T, cmap='gray')
            axes[0, i].set_title(f'模态 {i}')
            axes[0, i].axis('off')
            plt.colorbar(im, ax=axes[0, i], fraction=0.046, pad=0.04)
    
    # 显示标签
    im_label = axes[1, 0].imshow(label_3d[:, :, slice_idx].T, cmap='tab20')
    axes[1, 0].set_title('标签')
    axes[1, 0].axis('off')
    plt.colorbar(im_label, ax=axes[1, 0], fraction=0.046, pad=0.04)
    
    # 显示另外3个模态
    for i in range(1, 4):
        modal_idx = i + 3
        if modal_idx < data_4d.shape[3]:
            im = axes[1, i].imshow(data_4d[:, :, slice_idx, modal_idx].T, cmap='gray')
            axes[1, i].set_title(f'模态 {modal_idx}')
            axes[1, i].axis('off')
            plt.colorbar(im, ax=axes[1, i], fraction=0.046, pad=0.04)
    
    plt.suptitle(f'{subject_id} - Balanced数据切片 (z={slice_idx})')
    plt.tight_layout()
    
    # 保存图片
    output_dir = Path("./balanced_analysis_results")
    output_dir.mkdir(exist_ok=True)
    plt.savefig(output_dir / f'balanced_slices_{subject_id}.png', dpi=150, bbox_inches='tight')
    plt.show()

# 可视化第一个受试者的数据
if valid_balanced_subjects:
    print("\n📈 生成可视化...")
    plot_balanced_slices(balanced_data_4d, balanced_data_label, first_subject['subject_id'])

# ## 11. 批量检查所有受试者的balanced数据一致性

def batch_check_balanced_data(valid_subjects, max_check=None):
    """
    批量检查所有受试者的balanced数据一致性
    """
    if max_check:
        valid_subjects = valid_subjects[:max_check]
    
    print(f"\n🔄 批量检查 {len(valid_subjects)} 个受试者的Balanced数据...")
    print("=" * 80)
    
    consistency_results = []
    
    for subject in tqdm(valid_subjects, desc="检查进度"):
        try:
            # 加载数据头信息（不加载数据内容以节省内存）
            img_4d = nib.load(subject['balanced_4d_path'])
            label_3d = nib.load(subject['balanced_3d_labels_path'])
            
            # 记录形状信息
            result = {
                'subject_id': subject['subject_id'],
                '4d_shape': img_4d.shape,
                '3d_shape': label_3d.shape,
                'n_modalities': img_4d.shape[3] if len(img_4d.shape) == 4 else 0,
                'total_voxels': np.prod(label_3d.shape),
                'affine_match': np.allclose(img_4d.affine, label_3d.affine, rtol=1e-5)
            }
            
            consistency_results.append(result)
            
        except Exception as e:
            consistency_results.append({
                'subject_id': subject['subject_id'],
                'error': str(e)
            })
    
    return consistency_results

# 执行批量检查
print("\n开始批量检查数据一致性...")
consistency_results = batch_check_balanced_data(valid_balanced_subjects)

# 分析一致性结果
df_consistency = pd.DataFrame(consistency_results)

# 检查形状一致性
if '4d_shape' in df_consistency.columns:
    unique_4d_shapes = df_consistency['4d_shape'].value_counts()
    unique_3d_shapes = df_consistency['3d_shape'].value_counts()
    
    print("\n📊 数据形状一致性:")
    print(f"\n4D数据形状分布:")
    for shape, count in unique_4d_shapes.items():
        print(f"  {shape}: {count} 个受试者")
    
    print(f"\n3D标签形状分布:")
    for shape, count in unique_3d_shapes.items():
        print(f"  {shape}: {count} 个受试者")
    
    # 检查模态数一致性
    if 'n_modalities' in df_consistency.columns:
        unique_modalities = df_consistency['n_modalities'].value_counts()
        print(f"\n模态数分布:")
        for n_mod, count in unique_modalities.items():
            print(f"  {n_mod} 个模态: {count} 个受试者")

# 保存一致性检查结果
consistency_file = output_dir / 'balanced_consistency_check.csv'
df_consistency.to_csv(consistency_file, index=False)
print(f"\n💾 一致性检查结果已保存到: {consistency_file}")

# ## 12. 生成数据处理建议

print("\n" + "=" * 80)
print("📋 Balanced数据分析总结与处理建议")
print("=" * 80)

if valid_balanced_subjects:
    # 基于分析结果给出建议
    print("\n基于分析结果，以下是数据处理建议：")
    
    print("\n1. 📐 数据形状:")
    print(f"   • Balanced数据形状: {balanced_data_4d.shape}")
    print(f"   • 总体素数: {total_voxels:,}")
    print(f"   • 模态数: {balanced_data_4d.shape[3]}")
    
    print("\n2. 🏷️ 标签特征:")
    print(f"   • 唯一标签数: {len(unique_labels)}")
    print(f"   • 标签范围: {int(unique_labels.min())} - {int(unique_labels.max())}")
    print(f"   • 背景占比: {background_percentage:.2f}%")
    
    print("\n3. 💡 处理建议:")
    
    # 判断是否需要标签映射
    if len(unique_labels) == 52 and unique_labels.max() > 52:
        print("   ✓ 建议进行标签映射：将稀疏标签映射到0-51连续范围")
        print("     原因：标签值不连续，存在大于52的值")
    elif len(unique_labels) <= 52 and unique_labels.max() <= 51:
        print("   ✓ 标签已经在合理范围内，可能不需要映射")
    
    # 判断是否需要排除背景
    if background_percentage > 50:
        print(f"   ✓ 建议训练时考虑排除背景体素")
        print(f"     原因：背景占比{background_percentage:.1f}%过高")
    
    # 数据标准化建议
    print("\n   ✓ 数据预处理建议:")
    print("     • 使用StandardScaler进行特征标准化")
    print("     • 保持Fortran顺序(order='F')进行展平")
    print("     • 考虑数据增强策略提升模型泛化能力")
    
    print("\n4. 🚀 下一步操作:")
    print("   1) 运行数据展平处理")
    print("   2) 应用标签映射（如需要）")
    print("   3) 准备训练脚本")
    print("   4) 执行12折交叉验证")

# ## 13. 保存完整的分析报告

analysis_report = {
    "analysis_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_info": {
        "total_subjects": scan_results['total_subjects'],
        "valid_subjects": scan_results['valid_subjects'],
        "incomplete_subjects": scan_results['incomplete_subjects']
    },
    "data_characteristics": {
        "shape_4d": balanced_data_4d.shape if 'balanced_data_4d' in locals() else None,
        "shape_3d": balanced_data_label.shape if 'balanced_data_label' in locals() else None,
        "n_modalities": balanced_data_4d.shape[3] if 'balanced_data_4d' in locals() else None,
        "total_voxels": int(total_voxels) if 'total_voxels' in locals() else None,
        "unique_labels": int(len(unique_labels)) if 'unique_labels' in locals() else None,
        "label_range": [int(unique_labels.min()), int(unique_labels.max())] if 'unique_labels' in locals() else None,
        "background_percentage": float(background_percentage) if 'background_percentage' in locals() else None
    },
    "valid_subjects_list": valid_balanced_subjects,
    "recommendations": {
        "need_label_mapping": len(unique_labels) == 52 and unique_labels.max() > 52 if 'unique_labels' in locals() else None,
        "exclude_background": background_percentage > 50 if 'background_percentage' in locals() else None,
        "use_fortran_order": True
    }
}

report_file = output_dir / 'balanced_analysis_report.json'
with open(report_file, 'w', encoding='utf-8') as f:
    json.dump(analysis_report, f, ensure_ascii=False, indent=2)

print(f"\n💾 完整分析报告已保存到: {report_file}")

print("\n✅ Balanced数据分析完成！")
print(f"📁 所有结果保存在: {output_dir.absolute()}")

# ## 14. 生成数据展平处理的代码模板

print("\n" + "=" * 80)
print("📝 为Balanced数据生成处理代码模板")
print("=" * 80)

code_template = '''
# Balanced数据展平处理代码示例
# 基于分析结果自动生成

import numpy as np
import nibabel as nib
from pathlib import Path

def process_balanced_subject(subject_path):
    """处理单个受试者的balanced数据"""
    
    # 设置路径
    balanced_dir = Path(subject_path) / "balanced_output"
    output_dir = balanced_dir / "flattened"
    output_dir.mkdir(exist_ok=True)
    
    # 加载数据
    img_4d = nib.load(balanced_dir / "balanced_data_4d10000.nii.gz")
    label_3d = nib.load(balanced_dir / "balanced_labels_3d10000.nii.gz")
    
    data_4d = img_4d.get_fdata()
    labels = label_3d.get_fdata().astype(np.int32)
    
    # 展平数据（使用Fortran顺序）
    n_voxels = np.prod(labels.shape)
    n_modalities = data_4d.shape[3]
    
    features = data_4d.reshape(n_voxels, n_modalities, order='F')
    labels_flat = labels.flatten(order='F')
    
    # 创建非背景掩码
    region_mask = labels != 0
    
    # 保存展平数据
    np.save(output_dir / 'features.npy', features.astype(np.float32))
    np.save(output_dir / 'labels.npy', labels_flat.astype(np.int32))
    np.save(output_dir / 'region_mask.npy', region_mask)
    
    print(f"✓ 处理完成: {subject_path}")
    print(f"  Features: {features.shape}")
    print(f"  Labels: {labels_flat.shape}")
    
    return features, labels_flat, region_mask

# 使用示例
# subject_path = "/path/to/FOR_001"
# features, labels, mask = process_balanced_subject(subject_path)
'''

print("生成的处理代码模板：")
print(code_template)

template_file = output_dir / 'balanced_processing_template.py'
with open(template_file, 'w') as f:
    f.write(code_template)

print(f"\n💾 代码模板已保存到: {template_file}")

print("\n🎯 分析完成！请根据上述分析结果决定：")
print("   1. 是否需要进行标签映射")
print("   2. 选择合适的数据预处理策略")
print("   3. 确定模型训练参数")